# Module 03 — RAG (Retrieval-Augmented Generation)

In this notebook you will learn:
- What RAG is and how it solves the freshness problem
- How embeddings and vector databases work
- How to build a complete RAG pipeline from scratch


## 1. The Freshness Problem

LLMs are trained on a static snapshot of data. After training:
- They don't know about events after the cutoff date
- They can't access your private/company data
- Retraining is expensive and slow

**RAG solution:** At query time, retrieve relevant documents from an up-to-date knowledge base and inject them into the prompt as context.

```
User question
    → Search knowledge base
    → Retrieve relevant chunks
    → Inject into prompt
    → LLM generates answer grounded in retrieved facts
```


## 2. Embeddings

To search documents semantically (by meaning, not keyword), we convert text to **embeddings** — dense numerical vectors that capture meaning. Similar text → similar vectors.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a small, fast embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    'The cat sat on the mat.',
    'A feline rested on a rug.',
    'The stock market crashed today.',
]

embeddings = embedder.encode(sentences)
print(f'Embedding shape: {embeddings.shape}')  # (3, 384)

# Cosine similarity between sentences
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(embeddings)

print(f'\nSimilarity (cat vs feline): {sim[0][1]:.3f}')
print(f'Similarity (cat vs stocks): {sim[0][2]:.3f}')

## 3. Vector Database (ChromaDB)

A vector database stores embeddings and allows fast similarity search at scale.


In [ ]:
import chromadb

# Create an in-memory vector store (use PersistentClient for disk storage)
client = chromadb.Client()
collection = client.create_collection('knowledge_base')

# Sample knowledge base documents
documents = [
    'RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a language model.',
    'LoRA is a parameter-efficient fine-tuning technique that adds low-rank matrices to frozen model weights.',
    'ChromaDB is an open-source vector database designed for AI applications.',
    'Embeddings are dense vector representations of text that capture semantic meaning.',
    'The context window of an LLM determines how much text it can process in one pass.',
    'Fine-tuning adapts a pre-trained model to a specific task or domain using new training data.',
    'Vector similarity search finds documents whose embeddings are closest to a query embedding.',
    'Sentence transformers produce fixed-size embeddings from variable-length text.',
]

# Add documents to the collection
collection.add(
    documents=documents,
    ids=[f'doc_{i}' for i in range(len(documents))]
)

print(f'Knowledge base loaded with {collection.count()} documents')

## 4. Retrieval


In [ ]:
def retrieve(query: str, n_results: int = 3) -> list[str]:
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results['documents'][0]

query = 'How do I search documents by meaning?'
retrieved = retrieve(query)

print(f'Query: {query}\n')
for i, doc in enumerate(retrieved, 1):
    print(f'Result {i}: {doc}')

## 5. Full RAG Pipeline


In [ ]:
from transformers import pipeline

generator = pipeline('text-generation', model='distilgpt2', device='cpu')

def rag_answer(question: str) -> str:
    # Step 1: Retrieve relevant context
    context_docs = retrieve(question, n_results=2)
    context = ' '.join(context_docs)
    
    # Step 2: Build prompt with injected context
    prompt = f"""Context: {context}

Question: {question}
Answer:"""
    
    # Step 3: Generate answer
    result = generator(
        prompt,
        max_new_tokens=80,
        do_sample=False,
        truncation=True
    )
    return result[0]['generated_text']

print(rag_answer('What is retrieval augmented generation?'))

## Next Steps
Move on to `04_context_management/` to learn how to keep your knowledge base fresh and accurate over time.
